In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [8]:
!pip install mlflow dagshub xgboost -q

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import dagshub
from sklearn.preprocessing import LabelEncoder
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("ML_assn1")
dagshub_username = user_secrets.get_secret("username")

os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

repo_owner = dagshub_username
repo_name  = "ml_assn2"

mlflow.set_tracking_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
mlflow.set_registry_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
dagshub.init(repo_name=repo_name, repo_owner=repo_owner, mlflow=True)

Initialized MLflow to track repo "lkuch23/ml_assn2"

Repository lkuch23/ml_assn2 initialized!

In [9]:
MODEL_NAME = "IEEE_XGBoost_Best"
model_uri = "models:/IEEE_XGBoost_Best@production"
model = mlflow.xgboost.load_model(model_uri)

In [10]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test = test_transaction.merge(test_identity, on='TransactionID', how='left')

# Cleaning

In [11]:
missing_ratio = train.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.85].index
train.drop(cols_to_drop, axis=1, inplace=True)
test.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')

In [12]:
train.fillna(-999, inplace=True)
test.fillna(-999, inplace=True)

# Feature Engineering

In [13]:
train['hour'] = (train['TransactionDT'] // 3600) % 24
test['hour'] = (test['TransactionDT'] // 3600) % 24
train['day_of_week'] = (train['TransactionDT'] // (3600*24)) % 7
test['day_of_week'] = (test['TransactionDT'] // (3600*24)) % 7
train['amt_log'] = np.log1p(train['TransactionAmt'])
test['amt_log'] = np.log1p(test['TransactionAmt'])
train['amt_cents'] = train['TransactionAmt'] - train['TransactionAmt'].astype(int)
test['amt_cents'] = test['TransactionAmt'] - test['TransactionAmt'].astype(int)
train['amt_per_hour'] = train['TransactionAmt'] / (train['hour'] + 1)
test['amt_per_hour'] = test['TransactionAmt'] / (test['hour'] + 1)

In [14]:
for col in ['card1', 'card2', 'card3', 'card5']:
    if col in train.columns:
        grp = train.groupby(col)['TransactionAmt'].agg(['mean', 'std', 'count'])
        grp.columns = [f'{col}_amt_mean', f'{col}_amt_std', f'{col}_count']
        train = train.join(grp, on=col)
        test  = test.join(grp, on=col)

train.fillna(-999, inplace=True)
test.fillna(-999, inplace=True)

In [15]:
cat_cols = train.select_dtypes(include='object').columns
for col in cat_cols:
    if col in test.columns:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]]).astype(str)
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
    else:
        train[col] = train[col].astype(str)
        train[col] = train[col].factorize()[0]

# Feature Selection

In [16]:
target = 'isFraud'
ID = 'TransactionID'
common_cols = set(train.columns) & set(test.columns)
features = [c for c in common_cols if c not in [target, ID]]
features = sorted(features)
X = train[features].apply(pd.to_numeric, errors='coerce').fillna(-999)
y = train[target]
X_test = test[features].apply(pd.to_numeric, errors='coerce').fillna(-999)

# Submission

In [17]:
test_preds = model.predict_proba(X_test)[:, 1]

In [19]:
submission = pd.DataFrame({"TransactionID": test["TransactionID"], "isFraud": test_preds,})
output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)